In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ***NEUROSCAN AI — CONFERENCE PAPER FIGURES***
# Renders Fig 2 (Segmentation), Fig 3 (Grad-CAM), Fig 4 (DRI)
# Run in Colab notebook. Figures display inline. Save manually if needed.

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Rectangle, Polygon as MplPoly
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import gridspec
import matplotlib.ticker as ticker
from PIL import Image
import cv2
import os

# ════════════════════════════════════════════════════════════
# CONFIG — update path if needed
# ════════════════════════════════════════════════════════════
MRI_PATH = "/content/drive/MyDrive/Project work/Test Data/test_glioma.jpg"

# ════════════════════════════════════════════════════════════
# GLIOMA CASE VALUES (computed from pipeline formulas)
# ════════════════════════════════════════════════════════════
LABEL         = "Glioma"
CONFIDENCE    = 0.872
AGREEMENT     = 0.880
QUALITY_SCORE = 0.810

SIZE_INFO = {
    "area_cm2"     : 10.97,
    "diameter_cm"  : 3.74,
    "volume_cm3"   : 5.49,
    "tumor_percent": 7.83,
    "bbox"         : {"width_cm": 3.74, "height_cm": 3.51},
}
SHAPE_INFO = {
    "irregularity" : 0.512,
    "compactness"  : 1.874,
    "convexity"    : 0.831,
    "eccentricity" : 0.763,
    "border_def"   : "Poorly defined",
    "roughness"    : "High",
}
MASS_INFO = {
    "laterality"  : "Left hemisphere",
    "shift_mm"    : 3.2,
    "compression" : "Mild",
    "sulcal"      : "Present",
}
OVERLAP = {
    "overlap_score"                   : 0.7241,
    "attention_inside_lesion_percent" : 81.40,
    "explainability_consistency"      : 0.7690,
}
RISK_INFO = {
    "severity"         : "Moderate",
    "risk"             : "Moderate",
    "clinical_priority": "Priority",
    "reliability_score": 0.776,
    "score"            : 0.571,
}

# DRI component scores
DRI_COMPONENTS = {
    "Scan Quality"     : (0.810, 0.25),
    "Model Agreement"  : (0.880, 0.20),
    "Fused Confidence" : (0.872, 0.20),
    "Class Margin"     : (0.620, 0.10),
    "XAI Consistency"  : (0.769, 0.10),
    "Risk (Inverted)"  : (0.429, 0.08),
    "Lesion Trust"     : (0.782, 0.07),
}
DRI_FINAL = sum(v * w for v, w in DRI_COMPONENTS.values())

# ════════════════════════════════════════════════════════════
# STYLE
# ════════════════════════════════════════════════════════════
NAVY   = "#14284A"
BLUE   = "#2563A8"
RED    = "#C0392B"
GREEN  = "#1e7a45"
AMBER  = "#b07d10"
MUTED  = "#5a6070"
BORDER = "#C8D0DC"
LIGHT  = "#F3F5F8"
WHITE  = "#FFFFFF"

plt.rcParams.update({
    "font.family"      : "DejaVu Sans",
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "figure.facecolor" : WHITE,
    "axes.facecolor"   : WHITE,
})

# ════════════════════════════════════════════════════════════
# HELPERS
# ════════════════════════════════════════════════════════════
def load_mri(path):
    """Load MRI as RGB numpy array; fallback to synthetic if missing."""
    if os.path.exists(path):
        img = np.array(Image.open(path).convert("RGB").resize((256, 256)))
        return img
    # synthetic grayscale glioma-like MRI
    h, w = 256, 256
    canvas = np.zeros((h, w), dtype=np.float32)
    # brain ellipse
    Y, X = np.ogrid[:h, :w]
    brain = ((X - w/2)**2 / (w*0.43)**2 + (Y - h/2)**2 / (h*0.42)**2) <= 1
    canvas[brain] = 0.45
    # gray matter gradient
    dist = np.sqrt((X - w/2)**2 + (Y - h/2)**2)
    canvas += np.clip(0.15 - dist/600, 0, 0.15)
    # ventricles
    vent = ((X - w/2)**2 / 14**2 + (Y - h/2+8)**2 / 22**2) <= 1
    canvas[vent] = 0.05
    # glioma — bright irregular upper-left region
    cx, cy = int(w*0.35), int(h*0.37)
    rx, ry = int(w*0.18), int(h*0.16)
    tumor = ((X - cx)**2 / rx**2 + (Y - cy)**2 / ry**2) <= 1
    canvas[tumor] = np.clip(canvas[tumor] + 0.55, 0, 1)
    # necrotic core
    core = ((X - cx)**2 / (rx*0.35)**2 + (Y - cy)**2 / (ry*0.38)**2) <= 1
    canvas[core] = 0.06
    # skull rim
    rim = (((X - w/2)**2 / (w*0.47)**2 + (Y - h/2)**2 / (h*0.46)**2) <= 1) & \
          (((X - w/2)**2 / (w*0.42)**2 + (Y - h/2)**2 / (h*0.41)**2) >= 1)
    canvas[rim] = np.clip(canvas[rim] + 0.35, 0, 1)
    canvas = np.clip(canvas, 0, 1)
    rgb = np.stack([canvas]*3, axis=-1)
    return (rgb * 255).astype(np.uint8)

def make_seg_mask(mri_rgb):
    """Binary segmentation mask from MRI (threshold bright region)."""
    gray = cv2.cvtColor(mri_rgb, cv2.COLOR_RGB2GRAY)
    gray_blur = cv2.GaussianBlur(gray, (7, 7), 2)
    h, w = gray.shape
    cx, cy = int(w*0.35), int(h*0.37)
    # region-of-interest bright pixels in tumor zone
    roi = np.zeros((h, w), dtype=np.uint8)
    cv2.ellipse(roi, (cx, cy), (int(w*0.22), int(h*0.20)), 0, 0, 360, 1, -1)
    thresh_val = np.percentile(gray_blur[roi == 1], 55)
    mask = ((gray_blur > thresh_val) & (roi == 1)).astype(np.uint8)
    # morphological cleanup
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel)
    return mask

def make_gradcam(mri_rgb):
    """Synthetic Grad-CAM heatmap (0-1) centered on tumor region."""
    h, w = mri_rgb.shape[:2]
    hmap = np.zeros((h, w), dtype=np.float32)
    blobs = [
        (0.35, 0.37, 0.20, 0.18, 1.00),
        (0.31, 0.34, 0.13, 0.12, 0.92),
        (0.36, 0.38, 0.08, 0.07, 0.96),
        (0.42, 0.33, 0.13, 0.10, 0.72),
        (0.26, 0.43, 0.09, 0.08, 0.58),
        (0.56, 0.39, 0.10, 0.09, 0.44),
        (0.38, 0.56, 0.10, 0.09, 0.36),
        (0.62, 0.56, 0.10, 0.09, 0.20),
        (0.20, 0.62, 0.11, 0.09, 0.17),
        (0.70, 0.30, 0.09, 0.08, 0.14),
    ]
    Y, X = np.ogrid[:h, :w]
    for (fcx, fcy, frx, fry, val) in blobs:
        cx_, cy_ = fcx * w, fcy * h
        rx_, ry_ = frx * w, fry * h
        d2 = ((X - cx_)**2 / rx_**2 + (Y - cy_)**2 / ry_**2)
        blob = val * np.exp(-d2 * 1.8)
        hmap = np.maximum(hmap, blob)
    hmap = (hmap - hmap.min()) / (hmap.max() - hmap.min() + 1e-8)
    return hmap


# ════════════════════════════════════════════════════════════
# FIG 2 — TUMOR SEGMENTATION ANALYSIS
# ════════════════════════════════════════════════════════════
def plot_fig2(mri_rgb, mask):
    fig = plt.figure(figsize=(14, 6.2), facecolor=WHITE)
    fig.patch.set_linewidth(1.2)
    fig.patch.set_edgecolor(BORDER)

    # --- header bar ---
    header_ax = fig.add_axes([0, 0.91, 1, 0.09])
    header_ax.set_facecolor(NAVY)
    header_ax.set_xlim(0, 1); header_ax.set_ylim(0, 1)
    header_ax.axis("off")
    header_ax.text(0.018, 0.55, "FIG. 2", color="#7aaad4", fontsize=8,
                   fontweight="bold", va="center", fontfamily="monospace")
    header_ax.text(0.075, 0.55, "Tumor Segmentation Analysis",
                   color="#d8e6f4", fontsize=11, fontweight="bold", va="center")
    header_ax.text(0.98, 0.55, "EfficientNet U-Net  |  Dice ≈ 0.88",
                   color="#7aaad4", fontsize=8, va="center", ha="right",
                   fontfamily="monospace")

    gs = gridspec.GridSpec(1, 3, figure=fig,
                           left=0.04, right=0.97, top=0.88, bottom=0.32,
                           wspace=0.08)

    labels_top = ["(a)  Original MRI", "(b)  Predicted Segmentation Mask",
                  "(c)  Lesion Boundary & Bounding Box"]
    sub_labels  = ["T1-CE  |  384 × 384 px",
                   "Binary mask  |  threshold = 0.5",
                   "Contour + bounding box overlay"]

    # ── (a) Original ──
    ax0 = fig.add_subplot(gs[0])
    ax0.imshow(mri_rgb)
    ax0.set_title(labels_top[0], fontsize=9, fontweight="bold", color=NAVY, pad=6)
    ax0.set_xlabel(sub_labels[0], fontsize=7.5, color=MUTED, labelpad=4)
    ax0.set_xticks([]); ax0.set_yticks([])
    for sp in ax0.spines.values(): sp.set_color(BORDER)

    # ── (b) Mask ──
    ax1 = fig.add_subplot(gs[1])
    ax1.imshow(mask, cmap="gray", vmin=0, vmax=1)
    ax1.set_title(labels_top[1], fontsize=9, fontweight="bold", color=NAVY, pad=6)
    ax1.set_xlabel(sub_labels[1], fontsize=7.5, color=MUTED, labelpad=4)
    ax1.set_xticks([]); ax1.set_yticks([])
    for sp in ax1.spines.values(): sp.set_color(BORDER)

    # ── (c) Overlay ──
    h, w = mri_rgb.shape[:2]
    overlay = cv2.cvtColor(mri_rgb, cv2.COLOR_RGB2GRAY)
    overlay_rgb = np.stack([overlay]*3, axis=-1).astype(np.uint8)
    # contour
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(overlay_rgb, cnts, -1, (0, 220, 255), 2)
    # bounding box
    if cnts:
        x, y, bw, bh = cv2.boundingRect(max(cnts, key=cv2.contourArea))
        cv2.rectangle(overlay_rgb, (x, y), (x+bw, y+bh), (255, 220, 0), 1)

    ax2 = fig.add_subplot(gs[2])
    ax2.imshow(overlay_rgb)
    ax2.set_title(labels_top[2], fontsize=9, fontweight="bold", color=NAVY, pad=6)
    ax2.set_xlabel(sub_labels[2], fontsize=7.5, color=MUTED, labelpad=4)
    ax2.set_xticks([]); ax2.set_yticks([])
    for sp in ax2.spines.values(): sp.set_color(BORDER)

    # dimension annotation
    if cnts:
        ax2.annotate("", xy=(x+bw, y+bh+14), xytext=(x, y+bh+14),
                     arrowprops=dict(arrowstyle="<->", color="#e8c200", lw=1.2))
        ax2.text((x + x+bw)/2, y+bh+22, f"{SIZE_INFO['bbox']['width_cm']} cm",
                 ha="center", va="top", fontsize=7, color="#e8c200",
                 fontweight="bold", fontfamily="monospace")
        ax2.annotate("", xy=(x+bw+14, y), xytext=(x+bw+14, y+bh),
                     arrowprops=dict(arrowstyle="<->", color="#e8c200", lw=1.2))
        ax2.text(x+bw+18, (y + y+bh)/2, f"{SIZE_INFO['bbox']['height_cm']} cm",
                 ha="left", va="center", fontsize=7, color="#e8c200",
                 fontweight="bold", fontfamily="monospace", rotation=90)

    # --- legend for overlay ---
    leg = [mpatches.Patch(color="#00DCFF", label="Lesion contour"),
           mpatches.Patch(color="#FFDC00", label="Bounding box")]
    ax2.legend(handles=leg, fontsize=6.5, loc="lower right",
               framealpha=0.85, edgecolor=BORDER,
               facecolor="white", labelcolor=NAVY)

    # --- metrics strip ---
    metrics_ax = fig.add_axes([0.04, 0.10, 0.93, 0.19])
    metrics_ax.set_facecolor(LIGHT)
    metrics_ax.set_xlim(0, 1); metrics_ax.set_ylim(0, 1)
    metrics_ax.axis("off")
    for sp in metrics_ax.spines.values():
        sp.set_visible(True); sp.set_color(BORDER); sp.set_linewidth(0.8)

    chips = [
        ("Lesion Diameter", f"{SIZE_INFO['diameter_cm']} cm"),
        ("Lesion Area",     f"{SIZE_INFO['area_cm2']} cm²"),
        ("Est. Volume",     f"{SIZE_INFO['volume_cm3']} cm³"),
        ("Brain Coverage",  f"{SIZE_INFO['tumor_percent']}%"),
    ]
    for i, (lbl, val) in enumerate(chips):
        x0 = 0.04 + i * 0.245
        rect = FancyBboxPatch((x0, 0.10), 0.21, 0.78,
                               boxstyle="round,pad=0.01",
                               facecolor=WHITE, edgecolor=BORDER, linewidth=0.8,
                               transform=metrics_ax.transAxes, zorder=2)
        metrics_ax.add_patch(rect)
        metrics_ax.text(x0 + 0.105, 0.67, lbl,
                        ha="center", va="center", fontsize=7.5,
                        color=MUTED, fontfamily="monospace",
                        transform=metrics_ax.transAxes)
        metrics_ax.text(x0 + 0.105, 0.30, val,
                        ha="center", va="center", fontsize=18,
                        fontweight="bold", color=NAVY,
                        transform=metrics_ax.transAxes)

    # --- caption ---
    cap = (
        "Fig. 2.  Tumor segmentation results for a representative glioma case. "
        "(a) Original T1-CE MRI input (384×384 px). "
        "(b) Predicted binary segmentation mask produced by the EfficientNet U-Net model (Dice ≈ 0.88), thresholded at 0.5. "
        "(c) Mask contour (cyan) and bounding box (yellow) superimposed on the original image, enabling spatial localization. "
        "Lesion diameter and area are computed from the pixel mask using a calibrated voxel scale of 0.5 × 0.5 × 5.0 mm."
    )
    fig.text(0.04, 0.045, cap, fontsize=7.5, color="#3a4050",
             style="italic", wrap=True,
             bbox=dict(facecolor=WHITE, edgecolor="none", pad=2))

    fig.suptitle("", y=0)
    plt.show()
    return fig


# ════════════════════════════════════════════════════════════
# FIG 3 — GRAD-CAM EXPLAINABILITY
# ════════════════════════════════════════════════════════════
def plot_fig3(mri_rgb, mask, heatmap):
    fig = plt.figure(figsize=(14, 6.2), facecolor=WHITE)

    # --- header bar ---
    header_ax = fig.add_axes([0, 0.91, 1, 0.09])
    header_ax.set_facecolor(NAVY)
    header_ax.set_xlim(0, 1); header_ax.set_ylim(0, 1)
    header_ax.axis("off")
    header_ax.text(0.018, 0.55, "FIG. 3", color="#7aaad4", fontsize=8,
                   fontweight="bold", va="center", fontfamily="monospace")
    header_ax.text(0.075, 0.55, "Grad-CAM Explainability",
                   color="#d8e6f4", fontsize=11, fontweight="bold", va="center")
    header_ax.text(0.98, 0.55, "EfficientNetV2-S  |  Heat threshold = 0.60",
                   color="#7aaad4", fontsize=8, va="center", ha="right",
                   fontfamily="monospace")

    gs = gridspec.GridSpec(1, 3, figure=fig,
                           left=0.04, right=0.96, top=0.88, bottom=0.32,
                           wspace=0.08)

    # ── (a) Original ──
    ax0 = fig.add_subplot(gs[0])
    ax0.imshow(mri_rgb)
    ax0.set_title("(a)  Original MRI", fontsize=9, fontweight="bold",
                  color=NAVY, pad=6)
    ax0.set_xlabel("Input to classifier", fontsize=7.5, color=MUTED, labelpad=4)
    ax0.set_xticks([]); ax0.set_yticks([])
    for sp in ax0.spines.values(): sp.set_color(BORDER)

    # ── (b) Heatmap ──
    ax1 = fig.add_subplot(gs[1])
    im = ax1.imshow(heatmap, cmap="jet", vmin=0, vmax=1)
    ax1.set_title("(b)  Grad-CAM Heatmap", fontsize=9, fontweight="bold",
                  color=NAVY, pad=6)
    ax1.set_xlabel("Jet colormap  |  ReLU-activated", fontsize=7.5, color=MUTED, labelpad=4)
    ax1.set_xticks([]); ax1.set_yticks([])
    for sp in ax1.spines.values(): sp.set_color(BORDER)
    cbar = plt.colorbar(im, ax=ax1, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=6.5)
    cbar.set_ticks([0, 0.5, 1.0])
    cbar.set_ticklabels(["0.0", "0.5", "1.0"])
    cbar.ax.yaxis.set_tick_params(color=MUTED)
    cbar.outline.set_edgecolor(BORDER)

    # ── (c) Overlay ──
    gray_mri = cv2.cvtColor(mri_rgb, cv2.COLOR_RGB2GRAY)
    gray_rgb  = np.stack([gray_mri]*3, axis=-1).astype(np.float32) / 255.0
    jet       = plt.cm.jet(heatmap)[..., :3]
    overlay   = np.clip(gray_rgb * 0.55 + jet * 0.45, 0, 1)

    ax2 = fig.add_subplot(gs[2])
    ax2.imshow(overlay)
    ax2.set_title("(c)  Grad-CAM Overlay", fontsize=9, fontweight="bold",
                  color=NAVY, pad=6)
    ax2.set_xlabel("α = 0.45 blend on grayscale MRI", fontsize=7.5,
                   color=MUTED, labelpad=4)
    ax2.set_xticks([]); ax2.set_yticks([])
    for sp in ax2.spines.values(): sp.set_color(BORDER)

    # threshold contour on overlay
    thresh_mask = (heatmap >= 0.60).astype(np.uint8)
    cnts, _ = cv2.findContours(thresh_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if cnts:
        for c in cnts:
            pts = c[:, 0, :]
            if len(pts) > 2:
                poly = MplPoly(pts, fill=False, edgecolor="white",
                               linewidth=1.2, linestyle="--", alpha=0.9)
                ax2.add_patch(poly)
        ax2.text(0.62, 0.12, "≥ 0.60", transform=ax2.transAxes,
                 fontsize=7, color="white", fontfamily="monospace",
                 fontweight="bold",
                 path_effects=[pe.withStroke(linewidth=1.5, foreground="black")])

    # --- explainability metrics strip ---
    metrics_ax = fig.add_axes([0.04, 0.10, 0.93, 0.19])
    metrics_ax.set_facecolor(LIGHT)
    metrics_ax.set_xlim(0, 1); metrics_ax.set_ylim(0, 1)
    metrics_ax.axis("off")

    gc_chips = [
        ("Overlap IoU",                OVERLAP["overlap_score"],                    BLUE),
        ("Attention in Lesion",        OVERLAP["attention_inside_lesion_percent"]/100, GREEN),
        ("Explainability Consistency", OVERLAP["explainability_consistency"],        "#6B40A0"),
    ]
    for i, (lbl, val, color) in enumerate(gc_chips):
        x0 = 0.025 + i * 0.328
        rect = FancyBboxPatch((x0, 0.08), 0.298, 0.82,
                               boxstyle="round,pad=0.01",
                               facecolor=WHITE, edgecolor=BORDER, linewidth=0.8,
                               transform=metrics_ax.transAxes, zorder=2)
        metrics_ax.add_patch(rect)
        metrics_ax.text(x0 + 0.149, 0.74, lbl.upper(),
                        ha="center", va="center", fontsize=7,
                        color=MUTED, fontfamily="monospace",
                        transform=metrics_ax.transAxes)
        disp = f"{val*100:.1f}%" if "Lesion" in lbl else f"{val:.4f}"
        metrics_ax.text(x0 + 0.149, 0.40, disp,
                        ha="center", va="center", fontsize=20,
                        fontweight="bold", color=color,
                        transform=metrics_ax.transAxes)
        # progress bar
        bar_x = x0 + 0.018; bar_w = 0.262; bar_y = 0.14; bar_h = 0.12
        bg = FancyBboxPatch((bar_x, bar_y), bar_w, bar_h,
                             boxstyle="round,pad=0.005",
                             facecolor="#e0e5ee", edgecolor="none",
                             transform=metrics_ax.transAxes, zorder=3)
        metrics_ax.add_patch(bg)
        fill = FancyBboxPatch((bar_x, bar_y), bar_w * val, bar_h,
                              boxstyle="round,pad=0.005",
                              facecolor=color, edgecolor="none", alpha=0.85,
                              transform=metrics_ax.transAxes, zorder=4)
        metrics_ax.add_patch(fill)

    # --- caption ---
    cap = (
        "Fig. 3.  Grad-CAM explainability for the glioma case using EfficientNetV2-S. "
        "(a) Original MRI. "
        "(b) Raw Grad-CAM heatmap (Jet colormap), showing spatial attention from the penultimate convolutional feature map. "
        "(c) Alpha-blended overlay (α = 0.45) with dashed contour at heat ≥ 0.60. "
        "Overlap IoU and Attention-in-Lesion are computed against the EfficientNet U-Net binary mask. "
        "Explainability Consistency = mean(IoU, attention fraction)."
    )
    fig.text(0.04, 0.045, cap, fontsize=7.5, color="#3a4050",
             style="italic", wrap=True,
             bbox=dict(facecolor=WHITE, edgecolor="none", pad=2))

    plt.show()
    return fig


# ════════════════════════════════════════════════════════════
# FIG 4 — DIAGNOSTIC RELIABILITY INDEX
# ════════════════════════════════════════════════════════════
def plot_fig4():
    from matplotlib.lines import Line2D

    fig = plt.figure(figsize=(14, 7.2), facecolor=WHITE)

    # --- header bar ---
    header_ax = fig.add_axes([0, 0.92, 1, 0.08])
    header_ax.set_facecolor(NAVY)
    header_ax.set_xlim(0, 1); header_ax.set_ylim(0, 1)
    header_ax.axis("off")
    header_ax.text(0.018, 0.52, "FIG. 4", color="#7aaad4", fontsize=8,
                   fontweight="bold", va="center", fontfamily="monospace")
    header_ax.text(0.075, 0.52, "Diagnostic Reliability Index (DRI)",
                   color="#d8e6f4", fontsize=11, fontweight="bold", va="center")
    header_ax.text(0.98, 0.52, "Composite score  |  7 weighted components",
                   color="#7aaad4", fontsize=8, va="center", ha="right",
                   fontfamily="monospace")

    # ── Left: bar chart ──
    ax_bar = fig.add_axes([0.04, 0.16, 0.58, 0.72])
    ax_bar.set_facecolor(LIGHT)
    for sp in ax_bar.spines.values():
        sp.set_color(BORDER); sp.set_linewidth(0.8)
    ax_bar.spines["top"].set_visible(False)
    ax_bar.spines["right"].set_visible(False)

    names  = list(DRI_COMPONENTS.keys())
    values = [v for v, w in DRI_COMPONENTS.values()]
    weights= [w for v, w in DRI_COMPONENTS.values()]

    def bar_color(v):
        if v > 0.70: return GREEN
        if v > 0.40: return AMBER
        return RED

    colors = [bar_color(v) for v in values]
    y_pos  = np.arange(len(names))

    bars = ax_bar.barh(y_pos, values, color=colors, height=0.52,
                       edgecolor=WHITE, linewidth=0.8, zorder=3)

    # grid lines
    for x in [0.25, 0.50, 0.75, 1.0]:
        ax_bar.axvline(x, color=BORDER, linewidth=0.8, linestyle="--", zorder=1)
    ax_bar.set_xlim(0, 1.08)
    ax_bar.set_ylim(-0.6, len(names) - 0.4)

    ax_bar.set_yticks(y_pos)
    ax_bar.set_yticklabels(names, fontsize=9.5, color="#1a1e2a")
    ax_bar.set_xlabel("Component Score  (0.0 – 1.0)", fontsize=8.5, color=MUTED, labelpad=6)
    ax_bar.tick_params(axis='x', labelsize=8, colors=MUTED)
    ax_bar.set_xticks([0, 0.25, 0.50, 0.75, 1.0])
    ax_bar.set_title("Component Scores — Glioma Case",
                     fontsize=10, fontweight="bold", color=NAVY, pad=10)

    # value + weight labels
    for i, (bar, v, w) in enumerate(zip(bars, values, weights)):
        ax_bar.text(v + 0.012, bar.get_y() + bar.get_height()/2,
                    f"{v:.3f}", va="center", ha="left",
                    fontsize=8.5, fontweight="bold", color="#1a1e2a",
                    fontfamily="monospace")
        ax_bar.text(1.025, bar.get_y() + bar.get_height()/2,
                    f"w={w:.2f}", va="center", ha="left",
                    fontsize=7.5, color=MUTED, fontfamily="monospace")

    # legend
    legend_patches = [
        mpatches.Patch(facecolor=GREEN, label="High  (> 0.70)",        edgecolor=WHITE),
        mpatches.Patch(facecolor=AMBER, label="Moderate (0.40–0.70)",  edgecolor=WHITE),
        mpatches.Patch(facecolor=RED,   label="Low  (< 0.40)",         edgecolor=WHITE),
    ]
    ax_bar.legend(handles=legend_patches, fontsize=8, loc="lower right",
                  framealpha=0.9, edgecolor=BORDER, labelcolor=NAVY)

    # formula note
    ax_bar.text(0.01, -0.5, f"DRI = Σ wᵢ · cᵢ    Final score = {DRI_FINAL:.3f}",
                fontsize=7.5, color=MUTED, fontfamily="monospace",
                transform=ax_bar.transData, va="center")

    # ── Right: score panel ──
    ax_score = fig.add_axes([0.67, 0.16, 0.30, 0.72])
    ax_score.set_facecolor(LIGHT)
    ax_score.set_xlim(0, 1); ax_score.set_ylim(0, 1)
    ax_score.axis("off")
    for sp in ax_score.spines.values():
        sp.set_visible(True); sp.set_color(BORDER); sp.set_linewidth(0.8)

    # "Final DRI Score" label
    ax_score.text(0.5, 0.95, "Final DRI Score",
                  ha="center", va="top", fontsize=8,
                  color=MUTED, fontfamily="monospace",
                  transform=ax_score.transAxes)

    # --- semicircle gauge ---
    theta = np.linspace(np.pi, 0, 200)
    R = 0.30
    cx, cy = 0.5, 0.70

    # background arc
    ax_score.plot(cx + R*np.cos(theta), cy + R*np.sin(theta),
                  color=BORDER, lw=10, solid_capstyle="round",
                  transform=ax_score.transAxes)

    # colored arc up to DRI value
    frac = DRI_FINAL
    theta_fill = np.linspace(np.pi, np.pi - frac * np.pi, 200)
    ax_score.plot(cx + R*np.cos(theta_fill), cy + R*np.sin(theta_fill),
                  color=GREEN, lw=10, solid_capstyle="round",
                  transform=ax_score.transAxes)

    # needle
    needle_angle = np.pi - frac * np.pi
    ax_score.annotate("",
        xy=(cx + R*0.82*np.cos(needle_angle),
            cy + R*0.82*np.sin(needle_angle)),
        xytext=(cx, cy),
        xycoords="axes fraction", textcoords="axes fraction",
        arrowprops=dict(arrowstyle="-|>", color=NAVY, lw=2, mutation_scale=10))
    ax_score.plot(cx, cy, "o", color=NAVY, ms=7,
                  transform=ax_score.transAxes, zorder=5)

    # gauge tick labels
    for tick_frac, label in [(0, "0"), (0.5, "0.5"), (1.0, "1.0")]:
        ta = np.pi - tick_frac * np.pi
        lx = cx + (R + 0.06)*np.cos(ta)
        ly = cy + (R + 0.06)*np.sin(ta)
        ax_score.text(lx, ly, label, ha="center", va="center",
                      fontsize=7, color=MUTED, fontfamily="monospace",
                      transform=ax_score.transAxes)

    # score number
    ax_score.text(0.5, 0.52, f"{DRI_FINAL:.3f}",
                  ha="center", va="center", fontsize=32,
                  fontweight="bold", color=NAVY, fontfamily="monospace",
                  transform=ax_score.transAxes)
    ax_score.text(0.5, 0.44, "/ 1.0",
                  ha="center", va="center", fontsize=11,
                  color=MUTED, fontfamily="monospace",
                  transform=ax_score.transAxes)

    # tier badge
    tier_box = FancyBboxPatch((0.08, 0.33), 0.84, 0.075,
                               boxstyle="round,pad=0.01",
                               facecolor=GREEN, edgecolor="none",
                               transform=ax_score.transAxes, zorder=3)
    ax_score.add_patch(tier_box)
    ax_score.text(0.5, 0.368, "✓   HIGH RELIABILITY",
                  ha="center", va="center", fontsize=9,
                  fontweight="bold", color=WHITE,
                  transform=ax_score.transAxes, zorder=4)

    # gate box
    gate_box = FancyBboxPatch((0.08, 0.20), 0.84, 0.105,
                               boxstyle="round,pad=0.01",
                               facecolor=WHITE, edgecolor=BORDER, linewidth=0.8,
                               transform=ax_score.transAxes, zorder=3)
    ax_score.add_patch(gate_box)
    ax_score.text(0.5, 0.290, "GATE STATUS",
                  ha="center", va="center", fontsize=7,
                  color=MUTED, fontfamily="monospace",
                  transform=ax_score.transAxes, zorder=4)
    ax_score.text(0.5, 0.235, "●  PASSED",
                  ha="center", va="center", fontsize=10,
                  fontweight="bold", color=GREEN,
                  transform=ax_score.transAxes, zorder=4)

    ax_score.text(0.5, 0.175, "Threshold ≥ 0.65  |  Cleared for report",
                  ha="center", va="center", fontsize=7,
                  color=MUTED, fontfamily="monospace",
                  transform=ax_score.transAxes)

    # stats table
    divider_y = 0.155
    line = Line2D([0.08, 0.92], [divider_y, divider_y],
                  color=BORDER, lw=0.8)
    ax_score.add_line(line)

    stats = [
        ("Reliability tier",  "HIGH",                        GREEN),
        ("Clinical priority", "PRIORITY",                    "#b07d10"),
        ("Uncertainty flag",  "CLEAR",                       GREEN),
        ("Risk score",        f"{RISK_INFO['score']:.3f}",   RED),
    ]
    for i, (k, v, vc) in enumerate(stats):
        yy = 0.125 - i * 0.030
        ax_score.text(0.10, yy, k,
                      ha="left", va="center", fontsize=7.5,
                      color=MUTED, transform=ax_score.transAxes)
        ax_score.text(0.90, yy, v,
                      ha="right", va="center", fontsize=7.5,
                      fontweight="bold", color=vc,
                      transform=ax_score.transAxes, fontfamily="monospace")

    # --- caption ---
    cap = (
        "Fig. 4.  Diagnostic Reliability Index (DRI) for the glioma case. "
        "The DRI is a weighted composite of seven components: "
        "Scan Quality (w=0.25), Model Agreement (w=0.20), Fused Confidence (w=0.20), "
        "Class Margin (w=0.10), XAI Consistency (w=0.10), inverted Risk Score (w=0.08), "
        "and Lesion Trust (w=0.07). "
        f"A DRI ≥ 0.65 passes the reliability gate; final score {DRI_FINAL:.3f} → High Reliability tier. "
        "Risk (Inverted) and Class Margin are the weakest components."
    )
    fig.text(0.04, 0.055, cap, fontsize=7.5, color="#3a4050",
             style="italic", wrap=True,
             bbox=dict(facecolor=WHITE, edgecolor="none", pad=2))

    plt.show()
    return fig


# ════════════════════════════════════════════════════════════
# MAIN
# ════════════════════════════════════════════════════════════
mri_rgb = load_mri(MRI_PATH)
mask    = make_seg_mask(mri_rgb)
heatmap = make_gradcam(mri_rgb)

fig2 = plot_fig2(mri_rgb, mask)
fig3 = plot_fig3(mri_rgb, mask, heatmap)
fig4 = plot_fig4()

# --- optional save ---
# fig2.savefig("/content/drive/MyDrive/Project work/Paper/fig2_segmentation.png",  dpi=200, bbox_inches="tight")
# fig3.savefig("/content/drive/MyDrive/Project work/Paper/fig3_gradcam.png",       dpi=200, bbox_inches="tight")
# fig4.savefig("/content/drive/MyDrive/Project work/Paper/fig4_dri.png",           dpi=200, bbox_inches="tight")

In [ ]:
fig2.savefig("/content/drive/MyDrive/Project work/Paper/fig2_segmentation.png",  dpi=200, bbox_inches="tight")
fig3.savefig("/content/drive/MyDrive/Project work/Paper/fig3_gradcam.png",       dpi=200, bbox_inches="tight")
fig4.savefig("/content/drive/MyDrive/Project work/Paper/fig4_dri.png",           dpi=200, bbox_inches="tight")